In [2]:
"""
DTS-ESN (Diverse-Timescale Echo State Network) — closed-loop prediction of the Rulkov map.

Implementation follows:
  Tanaka, Matsumori, Yoshida, Aihara,
  "Reservoir computing with diverse timescales for prediction of multiscale dynamics",
  Phys. Rev. Research 4, L032014 (2022).

State update (Eqs. 1-3 of the paper, Task 2 / closed-loop variant):

    x(t+dt) = (I - A) x(t) + A * tanh( rho*W x(t) + gamma*Win u(t+dt) + zeta*Wfb y(t) )
    y(t+dt) = W_out x(t+dt)

with A = diag(alpha_1, ..., alpha_Nx) and alpha_i drawn from a reciprocal
(log-uniform) distribution on [alpha_min, alpha_max].

Paper's Rulkov / Task 2 hyperparameters (Fig. 5(a) caption):
    Nx = 400, gamma = 0.8, rho = 1, zeta = 1, beta = 1e-3,
    alpha_min = 10^(-6/9), alpha_max = 1
"""

import numpy as np
import matplotlib.pyplot as plt

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..',)))
from lib.utils.reservoirpy import (
    fit_scaler, transform_array, inverse_transform_array
)


# ==========================================================
# DTS-ESN CLASS
# ==========================================================
class DTSESN:
    """
    Diverse-Timescale Echo State Network with output feedback.

    Parameters
    ----------
    n_inputs, n_outputs, n_reservoir : int
    alpha_min, alpha_max : float
        Bounds of the log-uniform leak-rate distribution.
    spectral_radius : float
        Target spectral radius rho of the recurrent matrix W.
    input_scaling : float
        Scaling factor gamma applied to Win.
    fb_scaling : float
        Scaling factor zeta applied to Wfb. Set to 0 to disable feedback.
    density : float
        Fraction of nonzero entries in W (d in the paper, default 0.1).
    ridge : float
        Tikhonov regularization factor beta for the readout.
    seed : int
    """

    def __init__(
        self,
        n_inputs,
        n_outputs,
        n_reservoir=400,
        alpha_min=10 ** (-6 / 9),
        alpha_max=1.0,
        spectral_radius=1.0,
        input_scaling=0.8,
        fb_scaling=1.0,
        density=0.1,
        ridge=1e-3,
        seed=42,
    ):
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs
        self.n_reservoir = n_reservoir
        self.alpha_min = alpha_min
        self.alpha_max = alpha_max
        self.spectral_radius = spectral_radius
        self.input_scaling = input_scaling
        self.fb_scaling = fb_scaling
        self.density = density
        self.ridge = ridge
        self.rng = np.random.default_rng(seed)

        self._build_weights()
        self._build_leak_rates()
        self.reset_state()

        self.W_out = None  # set during training

    # ----------------------------------------------------------
    def _build_weights(self):
        """Create W (sparse, rescaled), Win, Wfb."""
        Nx = self.n_reservoir

        # Recurrent matrix: uniform [-1,1] with density d, rescaled to unit
        # spectral radius, then multiplied by spectral_radius.
        W = self.rng.uniform(-1.0, 1.0, size=(Nx, Nx))
        mask = self.rng.uniform(0.0, 1.0, size=(Nx, Nx)) < self.density
        W = W * mask

        eigvals = np.linalg.eigvals(W)
        current_sr = np.max(np.abs(eigvals))
        if current_sr > 0:
            W = W / current_sr
        self.W = self.spectral_radius * W

        # Input weights: uniform [-1,1], full connectivity.
        self.W_in = self.rng.uniform(-1.0, 1.0, size=(Nx, self.n_inputs))

        # Feedback weights: uniform [-1,1], full connectivity.
        self.W_fb = self.rng.uniform(-1.0, 1.0, size=(Nx, self.n_outputs))

    # ----------------------------------------------------------
    def _build_leak_rates(self):
        """Draw log10(alpha_i) ~ U[log10(alpha_min), log10(alpha_max)]."""
        log_min = np.log10(self.alpha_min)
        log_max = np.log10(self.alpha_max)
        log_alphas = self.rng.uniform(log_min, log_max, size=self.n_reservoir)
        self.alphas = 10.0 ** log_alphas  # shape (Nx,)

    # ----------------------------------------------------------
    def reset_state(self):
        self.x = np.zeros(self.n_reservoir)
        self.y = np.zeros(self.n_outputs)

    # ----------------------------------------------------------
    def _update(self, u, y_prev):
        """
        One reservoir update.

        u      : shape (n_inputs,)   current input
        y_prev : shape (n_outputs,)  previous output (for feedback)
        """
        pre = (
            self.W @ self.x
            + self.input_scaling * (self.W_in @ u)
            + self.fb_scaling * (self.W_fb @ y_prev)
        )
        self.x = (1.0 - self.alphas) * self.x + self.alphas * np.tanh(pre)
        return self.x

    # ----------------------------------------------------------
    def harvest_states(self, U, Y_teacher, washout=0):
        """
        Drive the reservoir with inputs U and teacher-forced feedback Y_teacher,
        returning the collected reservoir states after the washout.

        U         : (T, n_inputs)
        Y_teacher : (T, n_outputs)   — used for feedback on the NEXT step
        """
        T = U.shape[0]
        X = np.zeros((T, self.n_reservoir))
        self.reset_state()

        y_prev = np.zeros(self.n_outputs)
        for t in range(T):
            self._update(U[t], y_prev)
            X[t] = self.x
            # teacher forcing: feedback of the true output at time t
            y_prev = Y_teacher[t]

        return X[washout:]

    # ----------------------------------------------------------
    def fit(self, U, Y, washout=100):
        """
        Train the readout by ridge regression.

        U : (T, n_inputs)
        Y : (T, n_outputs)
        """
        X = self.harvest_states(U, Y, washout=washout)
        Y_target = Y[washout:]

        # Ridge regression: W_out = (X^T X + beta I)^(-1) X^T Y
        Nx = self.n_reservoir
        A = X.T @ X + self.ridge * np.eye(Nx)
        B = X.T @ Y_target
        self.W_out = np.linalg.solve(A, B).T  # shape (n_outputs, Nx)

        # Retain final training state for continuity if desired.
        return self

    # ----------------------------------------------------------
    def run_open_loop(self, U, Y_teacher=None):
        """
        Drive the reservoir with known inputs U (and known feedback Y_teacher
        if provided, else use the model's own output) and return predictions.
        Used for synchronization / warmup on the test set.
        """
        T = U.shape[0]
        preds = np.zeros((T, self.n_outputs))
        y_prev = self.y.copy()

        for t in range(T):
            self._update(U[t], y_prev)
            y_hat = self.W_out @ self.x
            preds[t] = y_hat
            if Y_teacher is not None:
                y_prev = Y_teacher[t]
            else:
                y_prev = y_hat

        self.y = y_prev
        return preds

    # ----------------------------------------------------------
    def run_closed_loop(self, n_steps, u0):
        """
        Autonomous rollout. Starting from the current reservoir state and
        an initial input u0, feed the model's own predictions back as both
        input (via Win) and feedback (via Wfb).

        u0 : (n_inputs,) — seed input for the first autonomous step.
        """
        preds = np.zeros((n_steps, self.n_outputs))
        u = np.asarray(u0).reshape(self.n_inputs)
        y_prev = self.y.copy()

        for t in range(n_steps):
            self._update(u, y_prev)
            y_hat = self.W_out @ self.x
            preds[t] = y_hat
            # For a pure autoregressive setup (n_inputs == n_outputs),
            # the next input is the current prediction.
            u = y_hat
            y_prev = y_hat

        return preds

In [ ]:

# ==========================================================
# LOAD DATA
# ==========================================================
dataset = np.loadtxt('../data/chaotic_data/rulkov_map.csv', delimiter=',')
dataset = dataset[:, 0]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(dataset[:], color='steelblue', linewidth=0.8)
ax.set_xlabel('Step')
ax.set_ylabel('Signal')
plt.tight_layout()
plt.show()


# ==========================================================
# PARAMETERS
# ==========================================================

# -------------------------
# Data split
# -------------------------
train_len = 3000
test_start = 3000
test_len = 5000

# -------------------------
# Warmups
# -------------------------
train_warmup = 100    # washout during training
test_warmup = 500     # synchronization length before closed-loop rollout

# -------------------------
# DTS-ESN hyperparameters (Tanaka et al. 2022, Fig. 5(a) Rulkov, Task 2)
# -------------------------
res_units = 400
alpha_min = 10 ** (-6 / 9)   # ≈ 0.2154
alpha_max = 1.0
spectral_rad = 1.0
input_scaling = 0.8
fb_scaling = 1.0
reservoir_density = 0.1
ridge_beta = 1e-3
seed = 42

# -------------------------
# Data normalization
# -------------------------
normalization = "zscore"


# ==========================================================
# DATA PREPARATION
# ==========================================================
data = dataset.reshape(-1, 1)

X_raw = data[:-1]
Y_raw = data[1:]

X_train_raw = X_raw[:train_len]
Y_train_raw = Y_raw[:train_len]

X_test_raw = X_raw[test_start:test_start + test_len]
Y_test_raw = Y_raw[test_start:test_start + test_len]

scaler = fit_scaler(X_train_raw, method=normalization)

X_train = transform_array(X_train_raw, scaler)
Y_train = transform_array(Y_train_raw, scaler)

X_test = transform_array(X_test_raw, scaler)
Y_test = transform_array(Y_test_raw, scaler)

if test_warmup >= test_len:
    raise ValueError("test_warmup must be smaller than test_len.")


# ==========================================================
# BUILD & TRAIN DTS-ESN
# ==========================================================
esn = DTSESN(
    n_inputs=1,
    n_outputs=1,
    n_reservoir=res_units,
    alpha_min=alpha_min,
    alpha_max=alpha_max,
    spectral_radius=spectral_rad,
    input_scaling=input_scaling,
    fb_scaling=fb_scaling,
    density=reservoir_density,
    ridge=ridge_beta,
    seed=seed,
)

esn.fit(X_train, Y_train, washout=train_warmup)


# ==========================================================
# TESTING (synchronize then closed-loop)
# ==========================================================
esn.reset_state()

# Synchronize the reservoir with the true test input while teacher-forcing
# the feedback. This is the analog of the "warmup" period in the open-loop
# version; it brings reservoir state onto the target attractor.
if test_warmup > 0:
    _ = esn.run_open_loop(
        X_test[:test_warmup],
        Y_teacher=Y_test[:test_warmup],
    )

pred_len = test_len - test_warmup

# Seed input for the first autonomous step: the true input at that time.
u0 = X_test[test_warmup]

Y_pred_scaled = esn.run_closed_loop(pred_len, u0=u0)

Y_true_scaled = Y_test[test_warmup:test_warmup + pred_len]

Y_pred = inverse_transform_array(Y_pred_scaled, scaler).ravel()
Y_true = inverse_transform_array(Y_true_scaled, scaler).ravel()

Y_test_original = Y_test_raw.ravel()


# ==========================================================
# ERROR METRICS
# ==========================================================
mse = np.mean((Y_true - Y_pred) ** 2)
rmse = np.sqrt(mse)
nrmse = rmse / np.std(Y_true)

print(f"Normalization      : {normalization}")
print(f"Train interval     : [0 : {train_len}]")
print(f"Test interval      : [{test_start} : {test_start + test_len}]")
print(f"Train warmup       : {train_warmup}")
print(f"Test warmup        : {test_warmup}")
print(f"Prediction length  : {pred_len}")
print(f"Reservoir units    : {res_units}")
print(f"Reservoir density  : {reservoir_density}")
print(f"Spectral radius    : {spectral_rad}")
print(f"alpha_min, alpha_max: {alpha_min:.4f}, {alpha_max:.4f}")
print(f"Input scaling (γ)  : {input_scaling}")
print(f"Feedback scaling(ζ): {fb_scaling}")
print(f"Ridge (β)          : {ridge_beta:.2e}")
print(f"MSE                : {mse:.10f}")
print(f"RMSE               : {rmse:.10f}")
print(f"NRMSE              : {nrmse:.10f}")


# ==========================================================
# VISUALIZATION
# ==========================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# --- main trajectory plot ---
ax = axes[0]
ax.plot(
    np.arange(test_len),
    Y_test_original,
    c='green',
    linewidth=1.2,
    label="Ground truth",
)
ax.axvline(
    test_warmup,
    linestyle="--",
    c='k',
    linewidth=1.0,
    label="Warmup end",
)
ax.plot(
    np.arange(test_warmup, test_warmup + pred_len),
    Y_pred,
    linestyle="--",
    c='red',
    linewidth=1.2,
    label="DTS-ESN closed-loop prediction",
)
ax.set_title(
    f"DTS-ESN closed-loop prediction | NRMSE={nrmse:.4f} | "
    f"N={res_units}, ρ={spectral_rad}, "
    f"α∈[{alpha_min:.3f},{alpha_max}], γ={input_scaling}, "
    f"ζ={fb_scaling}, β={ridge_beta:.0e}"
)
ax.set_xlabel("Step within chosen test window")
ax.set_ylabel("Signal")
ax.legend()

# --- leak-rate distribution plot ---
ax = axes[1]
ax.hist(np.log10(esn.alphas), bins=30, color='steelblue', edgecolor='k')
ax.set_xlabel(r"$\log_{10}\alpha_i$")
ax.set_ylabel("Count")
ax.set_title("Distribution of reservoir leak rates (log-uniform)")

plt.tight_layout()
plt.show()